# Ollama and the autoregressive loop



## Setup




In [ ]:
!date


Tue Jul 28 12:18:40 AM UTC 2026


### Install packages


In [ ]:
%%capture output_system_install
%%bash

# bash setup
fgrep -q '.bash_aliases' ~/.bashrc || {
  echo -e '\n\n\n[ -f ~/.bash_aliases ] && source ~/.bash_aliases\n' >> ~/.bashrc
}

{ cat << 'eof'
alias cls='clear'
alias dir='ls -la'
alias goTo='cd '
alias goUp='cd ..'
alias whereAmI='pwd'

alias me.group.id='id -g'
alias me.group.name='id -g -n'
alias me.groups='id -G'
alias me.groups.ids='id -G'
alias me.groups.names='id -G -n '
alias me.id='id -u'
alias me.name='id -u -n'

eof
} > ~/.bash_aliases

{ cat << 'eof'

export PATH='/root/.local/bin':$PATH
eof
} >> ~/.bashrc


# system package installs
tmux new -s update -d " \
  apt-get update ;\
  apt-get install -y zstd ;\
  apt-get install -y tree jq ncal less texlive-xetex pandoc ; \
  echo == Done ; \
  sleep 30
"


### Jupyter

In [ ]:
%%capture output_install_run_jupyter
%%bash

# jupyter install
tmux new -s jupyter-server -d " \
  pip install ipyaml jupyterlab ; \
  jupyter labextension disable @jupyterlab/apputils-extension:announcements ; \
  jupyter lab \
    --ip=127.0.0.1 \
    --port=8888 \
    --no-browser \
    --allow-root \
    --NotebookApp.token='' ; \
  echo == Done ; \
  sleep 30
"


### Ollama


In [ ]:
%%capture output_install_run_ollama
%%bash

# ollama service install and launch
tmux new -s ollama -d "\
  mkdir -p /tmp/ollama-logs/ ; \
  exec > /tmp/ollama-logs/ollama.log 2>&1 ; \
  until which zstd ; do sleep 1 ;done ; \
  curl -fsSL https://ollama.com/install.sh | sh ; \
  OLLAMA_KEEP_ALIVE=20m OLLAMA_FLASH_ATTENTION=1 ollama serve ; \
  echo == Done ; \
  sleep 10
"


### Ollama models


Create a model file to modify the existing model


In [ ]:
%%writefile Modelfile.gemma4
FROM gemma4:e4b
PARAMETER num_ctx 32768


Writing Modelfile.gemma4


In [ ]:
%%capture output_install_ollama_models
%%bash

# ollama models pull and load
tmux new -s ollama_models -d "\
  mkdir -p /tmp/ollama-logs/ ; \
  exec > /tmp/ollama-logs/ollama.models.log 2>&1 ; \
  until curl -s -I 127.0.0.1:11434 ; do date ; sleep 1 ; done ;\
  ollama create gemma4:e4b-32k -f Modelfile.gemma4 ;\
  ollama ps ; \
  echo llama3.1:8b gemma4:12b  | nice -n 19 ionice -c 3 xargs -n 1 -P 2 ollama pull ; \
  echo ; \
  echo == Done ; \
  sleep 10
"


## Modules, etc.


In [ ]:
%alias tree tree

from datetime import datetime, timezone
from time import sleep
from google.colab import output
import requests

print("Waiting for Jupyter to start")
for i in range(300):
  try:
    requests.head( "http://127.0.0.1:8888" )
    print()
    break
  except:
    print("=", end="")
  sleep(1)
print(f"{i} seconds")

print("Jupyter has started")
output.serve_kernel_port_as_window(8888)


Waiting for Jupyter to start

0 seconds
Jupyter has started
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Wait for Ollama to download a model


In [ ]:
%%bash
echo Waiting for a model
date
until curl -s -I 127.0.0.1:11434 ; do sleep 1 ; done
date
until ollama list | grep gemma4:e4b-32k ; do sleep 1 ; done
date


Waiting for a model
Tue Jul 28 12:19:27 AM UTC 2026
HTTP/1.1 200 OK
Content-Type: text/plain; charset=utf-8
Date: Tue, 28 Jul 2026 00:19:48 GMT
Content-Length: 17

Tue Jul 28 12:19:48 AM UTC 2026
qwen2.5-coder:7b-32k              83dcf5dab80b    8.1 GB    Less than a second ago    
Tue Jul 28 12:21:42 AM UTC 2026


## Using Ollama


When ready, click on the link to Jupyter Lab, open a terminal, and type this to interact with ollama chat:

```
ollama run qwen2.5-coder:7b-32k "what is the capital of France?"
```

Or you can run it here.






In [ ]:
!ollama run qwen2.5-coder:7b-32k "what is the capital of France?"

The capital of France is Paris.



In [ ]:
!date


Tue Jul 28 12:26:13 AM UTC 2026


## Timer 1


In [ ]:
# show a timer for 30 minutes
print("Timer 1")
for i in range(60*30):
  utc_now = datetime.now(timezone.utc)
  print(f"\r{utc_now.timetz().isoformat(timespec='seconds')} {'==' * (utc_now.second % 10)}", end='')
  sleep(1)


Timer 1
00:27:42+00:00 ====

KeyboardInterrupt: 

## Attach Google Drive for data


In [ ]:
from google.colab import drive
drive.mount(
  '/content/drive',
  readonly=True,
)

## Python tool calling


In [ ]:
%%bash
pip install ollama --break-system-packages


In [ ]:
import ollama
import os
